In [1]:
#@title Imports
# ============================================
# Cell 1
# Install + import dependencies (Google Colab)
# ============================================

!pip -q install opencv-python-headless tqdm numpy
!apt-get -qq update
!apt-get -qq install -y ffmpeg

import cv2
import numpy as np
import os
import csv
import subprocess
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

print("Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment ready.


In [ ]:
#@title Load Tracks CSV
# ============================================
# Cell 2
# Colour palette + CSV loading helpers
# ============================================

_PALETTE = [
    (255,  80,  80), (80,  255,  80), (80,   80, 255), (255, 255,  80),
    (255,  80, 255), (80,  255, 255), (255, 160,  80), (160, 255,  80),
    (80,  160, 255), (255,  80, 160), (160,  80, 255), (80,  255, 160),
    (200, 200,  80), (200,  80, 200), (80,  200, 200), (255, 140,  40),
    (40,  255, 140), (140,  40, 255), (255,  40, 140), (40,  140, 255),
]


def cell_color(track_id):
    return _PALETTE[track_id % len(_PALETTE)]


def _parse_contour(contour_str):
    """
    Parse a "x1:y1;x2:y2;..." string (as written by stentorDetect2's
    export_tracks_csv) back into an OpenCV contour array.

    Returns None if the string is empty.
    """
    if not contour_str:
        return None

    pts = []
    for pair in contour_str.split(";"):
        if not pair:
            continue
        x_str, y_str = pair.split(":")
        pts.append((int(round(float(x_str))), int(round(float(y_str)))))

    if not pts:
        return None

    return np.array(pts, dtype=np.int32).reshape(-1, 1, 2)


def load_tracks_csv(csv_path, track_ids=None):
    """
    Load a tracks CSV (TRACK_ID, FRAME, POSITION_X, POSITION_Y, contour_points)
    as produced by stentorDetect2's export_tracks_csv.

    Args:
        csv_path   : path to CSV file
        track_ids  : optional iterable of track IDs to keep. If None, every
                     track in the file is kept.

    Returns:
        frame_tracks : dict mapping frame -> list of
                        {"track_id", "centroid", "contour"}
        n_frames     : 1 + max frame index seen in the CSV
        seen_ids     : sorted list of track IDs actually present after filtering
    """

    keep = set(int(t) for t in track_ids) if track_ids is not None else None

    frame_tracks = {}
    seen_ids = set()
    max_frame = -1

    with open(csv_path, "r", newline="") as fh:
        reader = csv.DictReader(fh)

        for row in reader:
            tid = int(row["TRACK_ID"])

            if keep is not None and tid not in keep:
                continue

            frame = int(row["FRAME"])
            cx = float(row["POSITION_X"])
            cy = float(row["POSITION_Y"])
            contour = _parse_contour(row.get("contour_points", ""))

            frame_tracks.setdefault(frame, []).append({
                "track_id": tid,
                "centroid": (cx, cy),
                "contour": contour
            })

            seen_ids.add(tid)
            max_frame = max(max_frame, frame)

    if keep is not None:
        missing = keep - seen_ids
        if missing:
            print(f"Warning: track IDs not found in CSV: {sorted(missing)}")

    return frame_tracks, max_frame + 1, sorted(seen_ids)

In [ ]:
#@title Draw Overlay + Trail
# ============================================
# Cell 3
# Overlay drawing: contour + centroid + label + trail
# ============================================


def _draw_trail(overlay, history, color, up_to_frame, trail_length,
                fade=True, thickness=2, font_scale=0.5):
    """
    Draw a fading trail of past centroids for one track onto an RGBA overlay.
    """

    pts = [
        (f, xy) for f, xy in history
        if f <= up_to_frame
    ]

    if trail_length is not None:
        cutoff = up_to_frame - trail_length
        pts = [(f, xy) for f, xy in pts if f > cutoff]

    if len(pts) < 2:
        return

    n = len(pts)

    for i in range(n - 1):
        f0, (x0, y0) = pts[i]
        f1, (x1, y1) = pts[i + 1]

        if fade:
            age_fraction = (i + 1) / n
            alpha = int(60 + 180 * age_fraction)
        else:
            alpha = 255

        cv2.line(
            overlay,
            (int(x0), int(y0)),
            (int(x1), int(y1)),
            color + (alpha,),
            thickness,
            cv2.LINE_AA
        )

    # Return the final visible point so the caller can place a label
    return pts[-1][1]


def draw_overlay_with_trail(frame_entries, track_histories, frame_idx, height, width,
                            trail_length=30, fade_trail=True,
                            contour_thickness=2, trail_thickness=2,
                            centroid_radius=4, font_scale=0.6,
                            laser_on_frame=None,
                            laser_off_frame=None):
    """
    Draw current-frame contours/centroids/labels plus per-track trails onto
    a transparent RGBA overlay frame.

    Returns:
        uint8 RGBA image
    """

    overlay = np.zeros((height, width, 4), dtype=np.uint8)

    laser_active = (
        laser_on_frame is not None and
        laser_off_frame is not None and
        laser_on_frame <= frame_idx <= laser_off_frame
    )
    _draw_laser_indicator(
        overlay,
        laser_active,
        width,
        height
    )

    # Trails first, so current-frame markers draw on top.
    for tid, history in track_histories.items():
        color = cell_color(tid)

        last_pt = _draw_trail(
            overlay,
            history,
            color,
            frame_idx,
            trail_length=trail_length,
            fade=fade_trail,
            thickness=trail_thickness,
        )

        # Label the end of the trail
        if last_pt is not None:
            x, y = map(int, last_pt)
            cv2.putText(
                overlay,
                str(tid),
                (x + 6, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                color + (255,),
                1,
                cv2.LINE_AA,
            )

    for entry in frame_entries:
        tid = entry["track_id"]
        cx, cy = entry["centroid"]
        contour = entry["contour"]

        color = cell_color(tid) + (255,)

        if contour is not None:
            pts = contour.reshape(-1, 2)

            for i in range(len(pts)):
                p1 = tuple(pts[i].astype(int))
                p2 = tuple(pts[(i + 1) % len(pts)].astype(int))

                cv2.line(overlay, p1, p2, color, contour_thickness, cv2.LINE_AA)

        cv2.circle(overlay, (int(cx), int(cy)), centroid_radius, color, -1)

        cv2.putText(
            overlay, str(tid),
            (int(cx) + 6, int(cy) - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale, color, 1, cv2.LINE_AA
        )

    return overlay

def _draw_laser_indicator(overlay, active, width, height):
    """
    Draw laser activity indicator in the top-right corner.

    Args:
        overlay : RGBA overlay image
        active  : whether laser is currently active
        width   : frame width
        height  : frame height
    """

    if not active:
        return

    # Indicator location
    x = int(round(width - (0.2 * width)))
    y = int(round(0.2 * height))
    fsize = 3

    # Label
    cv2.putText(
        overlay,
        "LIGHT ON",
        (x, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        fsize,
        (0, 0, 255, 0),
        2,
        cv2.LINE_AA
    )

In [ ]:
#@title Main Pipeline
# ============================================
# Cell 4
# Render trail overlay video
# ============================================


def render_trail_overlay(video_path, csv_path, track_ids, overlay_video,
                         trail_length=30, fade_trail=True,
                         contour_thickness=2, trail_thickness=2,
                         centroid_radius=4, font_scale=0.6,
                         laser_on_time=None,
                         laser_off_time=None):
    """
    Build an overlay MP4 that shows the selected tracks' contours, labels,
    and fading trail paths on top of the original video.

    Args:
        video_path      : path to source MP4
        csv_path        : path to tracks CSV (from stentorDetect2's export_tracks_csv)
        track_ids       : list of TRACK_ID values to render, or None for all
        overlay_video   : output MP4 path
        trail_length    : number of past frames to include in each trail
                          (None = full history since track start)
        fade_trail      : fade older trail segments so motion direction is clear
        contour_thickness, trail_thickness, centroid_radius, font_scale :
                          drawing style knobs
    """

    print("Loading tracks CSV...")
    frame_tracks, n_frames_csv, seen_ids = load_tracks_csv(csv_path, track_ids=track_ids)
    print(f"Tracks kept: {seen_ids}")

    if not seen_ids:
        raise ValueError("No matching tracks found in CSV - check TRACK_IDS.")

    # Per-track centroid history, sorted by frame, built once up front.
    track_histories = {tid: [] for tid in seen_ids}
    for frame, entries in frame_tracks.items():
        for e in entries:
            track_histories[e["track_id"]].append((frame, e["centroid"]))

    for tid in track_histories:
        track_histories[tid].sort(key=lambda p: p[0])

    # -----------------------------------------
    # Load video metadata
    # -----------------------------------------

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    # Convert laser timing from seconds to frame indices
    laser_on_frame = None
    laser_off_frame = None

    if laser_on_time is not None:
        laser_on_frame = int(round(laser_on_time * fps))

    if laser_off_time is not None:
        laser_off_frame = int(round(laser_off_time * fps))

    cap.release()

    n_frames = max(frame_count, n_frames_csv)

    print(f"Video: {width}x{height}, {frame_count} frames @ {fps:.2f} fps")
    print(f"CSV covers {n_frames_csv} frames")

    # -----------------------------------------
    # Render overlay frames
    # -----------------------------------------

    temp_dir = "trail_overlay_frames_temp"
    os.makedirs(temp_dir, exist_ok=True)

    for frame_idx in tqdm(range(n_frames), desc="Rendering trail overlay"):
        entries = frame_tracks.get(frame_idx, [])

        overlay_frame = draw_overlay_with_trail(
            entries,track_histories,frame_idx,height,
            width,trail_length=trail_length,fade_trail=fade_trail,
            contour_thickness=contour_thickness,trail_thickness=trail_thickness,
            centroid_radius=centroid_radius,font_scale=font_scale,
            laser_on_frame=laser_on_frame,laser_off_frame=laser_off_frame
        )

        cv2.imwrite(
            os.path.join(temp_dir, f"frame_{frame_idx:05d}.png"),
            overlay_frame
        )

    # -----------------------------------------
    # Composite overlay using ffmpeg
    # -----------------------------------------

    overlay_pattern = os.path.join(temp_dir, "frame_%05d.png")

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-framerate", str(fps),
        "-i", overlay_pattern,
        "-filter_complex", "[1]format=rgba[ovr];[0][ovr]overlay",
        "-c:a", "copy",
        overlay_video
    ]

    subprocess.run(cmd, check=True)

    print(f"Saved overlay -> {overlay_video}")

In [ ]:
#@title I/O
# ============================================
# Cell 5
# Notebook inputs + run
# ============================================

# -----------------------------
# Input / output files
# -----------------------------

VIDEO_PATH = "/content/C1_20260804_133401.mp4"      # source video
CSV_PATH   = "/content/test.csv"        # tracks CSV from stentorDetect2
OVERLAY_PATH = "/content/test.mp4"

LASER_ON_TIME = 5
LASER_OFF_TIME = 7

# -----------------------------
# Track selection
# -----------------------------
# List the TRACK_ID values (from the CSV's TRACK_ID column) you want to show.
# Set to None to show every track found in the CSV.

TRACK_IDS = [0, 1, 2]

# -----------------------------
# Trail appearance
# -----------------------------

TRAIL_LENGTH = 30          # number of past frames drawn in the trail (None = full history)
FADE_TRAIL = True          # older trail segments become more transparent
CONTOUR_THICKNESS = 4
TRAIL_THICKNESS = 5
CENTROID_RADIUS = 4
FONT_SCALE = 4

# -----------------------------
# Run
# -----------------------------

render_trail_overlay(
    video_path=VIDEO_PATH,
    csv_path=CSV_PATH,
    track_ids=TRACK_IDS,
    overlay_video=OVERLAY_PATH,
    trail_length=TRAIL_LENGTH,
    fade_trail=FADE_TRAIL,
    contour_thickness=CONTOUR_THICKNESS,
    trail_thickness=TRAIL_THICKNESS,
    centroid_radius=CENTROID_RADIUS,
    font_scale=FONT_SCALE,
    laser_on_time=LASER_ON_TIME,
    laser_off_time=LASER_OFF_TIME,
)

Loading tracks CSV...
Tracks kept: [0, 1, 2]
Video: 3856x2180, 77 frames @ 7.64 fps
CSV covers 77 frames


Rendering trail overlay: 100%|██████████| 77/77 [00:17<00:00,  4.47it/s]


Saved overlay -> /content/test.mp4
